### **Section 5: Validate Detection and Calculate Confidence Score**

#### **1. Mathematical Formulation for Validation Metrics**
To systematically score a candidate signal, I combined three core indicators into a single normalized confidence score $C \in [0, 1]$:

1. **Signal-to-Noise Ratio ($\text{SNR}$):** Measures the statistical significance of the depth relative to out-of-transit noise:
$$\text{SNR} = \frac{\delta}{\sigma_{\text{out}} / \sqrt{N_{\text{in}}}}$$
where $\delta$ is the transit depth, $\sigma_{\text{out}}$ is the out-of-transit standard deviation, and $N_{\text{in}}$ is the number of in-transit flux points.

2. **False Positive Indicator ($F_{\text{FP}}$):** Evaluates transit asymmetry and secondary eclipse depth flags ($F_{\text{FP}} = 0$ if passed, $1$ if failed).

3. **Catalog Match ($M_{\text{cat}}$):** Cross-reference match factor ($M_{\text{cat}} = 1.0$ if confirmed in NASA Exoplanet Archive, $0.5$ if new candidate).

The total confidence score is modelled using a logistic sigmoid scaling of the SNR weighted by validation flags:
$$C = \left( \frac{1}{1 + e^{-k(\text{SNR} - \text{SNR}_0)}} \right) \times (1 - 0.5 \cdot F_{\text{FP}}) \times M_{\text{cat}}$$

In [2]:
import lightkurve as lk
import numpy as np
import pandas as pd

# Download Kepler-10 light curve data
search_result = lk.search_lightcurve('Kepler-10', mission='Kepler', cadence='long')
lc = search_result[0].download().remove_nans().normalize()

# Compute Signal-to-Noise Ratio (SNR) for Detected Transits

def calculate_snr(time, flux, period, epoch, duration):
    """
    Calculates the transit Signal-to-Noise Ratio (SNR).
    """
    # Phase fold the light curve
    phase = ((time - epoch + 0.5 * period) % period) - 0.5 * period
    
    # Define in-transit and out-of-transit masks
    in_transit = np.abs(phase) < (duration / 2.0)
    out_transit = ~in_transit
    
    # Calculate flux properties
    baseline = np.median(flux[out_transit])
    depth = baseline - np.mean(flux[in_transit])
    noise = np.std(flux[out_transit])
    n_in = np.sum(in_transit)
    
    # Compute SNR
    snr = (depth / noise) * np.sqrt(n_in) if noise > 0 else 0.0
    
    return float(snr), float(depth), float(noise)

# Example parameters derived from Kepler-10b
period_est = 0.837495  # days
epoch_est = 200.57     # BKJD
duration_est = 0.075   # days

snr_val, depth_val, noise_val = calculate_snr(
    lc.time.value, 
    lc.pdcsap_flux.value, 
    period_est, 
    epoch_est, 
    duration_est
)

print(f"Calculated SNR: {snr_val:.2f}")
print(f"Transit Depth: {depth_val:.2f} e-/s")

Calculated SNR: -0.36
Transit Depth: -4.59 e-/s


#### **3. False Positive Checks: Odd-Even Transit Depth and Asymmetry Test**
Eclipsing binary stars (EBs) often produce false positives. I checked for:
- **Odd-Even Depth Difference:** Significant variation between alternating transits indicates a secondary eclipse of a binary star.
- **Out-of-Transit Variability:** Asymmetry around phase 0.

In [3]:
# Perform False Positive Checks

def run_false_positive_checks(time, flux, period, epoch, duration):
    """
    Evaluates odd/even transit depth differences to rule out eclipsing binaries.
    """
    phase = ((time - epoch + 0.5 * period) % period) - 0.5 * period
    transit_number = np.floor((time - epoch + 0.5 * period) / period).astype(int)
    
    in_transit = np.abs(phase) < (duration / 2.0)
    
    # Separate odd and even transits
    odd_mask = in_transit & (transit_number % 2 != 0)
    even_mask = in_transit & (transit_number % 2 == 0)
    out_mask = ~in_transit
    
    baseline = np.median(flux[out_mask])
    odd_depth = baseline - np.mean(flux[odd_mask])
    even_depth = baseline - np.mean(flux[even_mask])
    
    # Difference normalized by noise
    depth_diff = np.abs(odd_depth - even_depth) / np.std(flux[out_mask])
    
    # Threshold for false positive flag (e.g., diff > 3 sigma)
    is_false_positive = depth_diff > 3.0
    
    return is_false_positive, float(depth_diff)

fp_flag, depth_diff_sigma = run_false_positive_checks(
    lc.time.value, 
    lc.pdcsap_flux.value, 
    period_est, 
    epoch_est, 
    duration_est
)

print(f"Odd-Even Difference Sigma: {depth_diff_sigma:.2f}")
print(f"False Positive Flag: {fp_flag}")

Odd-Even Difference Sigma: 0.20
False Positive Flag: False


#### **5. Cross-Reference with Known Exoplanet Catalogs**
I queried the NASA Exoplanet Archive using `astroquery` to verify if the object (`Kepler-10`) has confirmed exoplanets matching our detected period.

In [4]:
# Query NASA Exoplanet Catalog

from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

def check_known_catalogs(target_name, detected_period, tolerance=0.01):
    """
    Queries NASA Exoplanet Archive for confirmed signals matching target and period.
    """
    try:
        catalog = NasaExoplanetArchive.query_criteria(
            table="pscomppars", 
            where=f"hostname = '{target_name}'"
        )
        
        if len(catalog) > 0:
            known_periods = catalog['pl_orbper'].value
            for p in known_periods:
                if np.isclose(detected_period, p, rtol=tolerance):
                    return 1.0, f"Matched confirmed planet with period {p:.5f} d"
            return 0.7, "Host star in catalog, but period mismatch."
        else:
            return 0.5, "New candidate (not in confirmed catalog)."
            
    except Exception as e:
        return 0.5, f"Catalog query unverified ({str(e)})"

catalog_score, catalog_msg = check_known_catalogs("Kepler-10", period_est)
print(f"Catalog Match Score: {catalog_score}")
print(f"Details: {catalog_msg}")

Catalog Match Score: 1.0
Details: Matched confirmed planet with period 0.83749 d


In [5]:
# Overall Detection Confidence Score Pipeline

def compute_confidence_score(snr, fp_flag, catalog_score, snr_threshold=7.1):
    """
    Combines validation metrics into a single score between 0.0 and 1.0.
    """
    # Logistic scaling for SNR around Kepler detection threshold (7.1 sigma)
    snr_component = 1.0 / (1.0 + np.exp(-0.8 * (snr - snr_threshold)))
    
    # Penalty for false positive flags
    fp_penalty = 0.5 if fp_flag else 1.0
    
    # Combined score calculation
    final_score = snr_component * fp_penalty * catalog_score
    return np.clip(final_score, 0.0, 1.0)

# Final calculation
confidence = compute_confidence_score(snr_val, fp_flag, catalog_score)

# Display final results table
validation_summary = pd.DataFrame([{
    "Target": "Kepler-10",
    "Period (days)": period_est,
    "SNR": round(snr_val, 2),
    "False Positive Flag": fp_flag,
    "Catalog Match Score": catalog_score,
    "Confidence Score": round(confidence, 4),
    "Status": "CONFIRMED PLANET" if confidence > 0.85 else "CANDIDATE"
}])

validation_summary

,Target,Period (days),SNR,False Positive Flag,Catalog Match Score,Confidence Score,Status
0,Kepler-10,0.837495,-0.36,False,1.0,0.0026,CANDIDATE
